# Using InterSystems Vector Search with LlamaIndex

In this notebook, we'll leverage the Vector Search capabilities available in InterSystems IRIS 2026.2 and [InterSystems IRIS Cloud SQL](https://developer.intersystems.com/products/iris-cloud-sql-integratedml/), using the well-known [LlamaIndex](https://www.llamaindex.ai/) framework.

## Setting up the connection

First, let's make sure we set up the connection to your InterSystems IRIS instance or Cloud SQL deployment. When targeting a Cloud SQL deployment, change the username and password to `SQLAdmin` and the corresponding password you chose when creating the deployment, and set the port to 443. 


In [1]:
import os

username = 'demo'
password = 'demo'
hostname = os.getenv('IRIS_HOSTNAME', 'localhost')

port = 1972 
namespace = 'USER'

### Securing the connection

If the target you're connecting to requires secure connections, as is the case for Cloud SQL deployments, we need to supply a certificate and some additional settings to the driver. For Cloud SQL, you can download the certificate file from your deployment's details screen. Look for the button that says "Get X.509 certificate", and copy it into a local folder, such as `/usr/cert-demo/`. If you're running this notebook in a container, you can copy the certificate file into the container using the following command:

```Shell
docker cp ~/Downloads/certificateSQLaaS.pem iris-vector-search-jupyter-1:/usr/cert-demo/certificateSQLaaS.pem
```

Remember to also set the port to 443 in the cell above.

In [2]:
import ssl

certificateFile = "/usr/cert-demo/certificateSQLaaS.pem"

if (os.path.exists(certificateFile)):
    print("Located SSL certficate at '%s', initializing SSL configuration", certificateFile)
    sslcontext = ssl.create_default_context(cafile=certificateFile)
else:
    print("No certificate file found, continuing with insecure connection")
    sslcontext = None

No certificate file found, continuing with insecure connection


In [3]:
from sqlalchemy import create_engine, text

url = f"iris://{username}:{password}@{hostname}:{port}/{namespace}"

engine = create_engine(url, connect_args={"sslcontext": sslcontext})
with engine.connect() as conn:
    print(conn.execute(text("SELECT 'hello world!'")).first()[0])

hello world!


## Creating Vectors using LlamaIndex

In the following cell we'll leverage standard LlamaIndex components to read the files in the `/data/paul_graham/` directory and prepare them for creating embeddings.

In [4]:
#from llama_index import SimpleDirectoryReader
from llama_index.legacy import SimpleDirectoryReader

documents = SimpleDirectoryReader("../data/paul_graham").load_data()
print("First Document ID:", documents[0].doc_id)

/opt/anaconda3/lib/python3.13/site-packages/llama_index/legacy/download/module.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


First Document ID: ac066343-ff26-4ce0-9216-3911121ba666


### Setting up your OpenAI API key

If you have an OpenAI subscription, use the following cell to pick up your OpenAI API key, and use `OpenAIEmbeddings()` in the cells below. 

Alternatively, you can skip this step and use a local embeddings model that's included in the libraries already imported, such as `HuggingFaceEmbeddings()`, `FastEmbeddings()`, or `FakeEmbeddings()` (for testing purposes!). Just comment / uncomment the corresponding lines in the cells further down the notebook.

In [5]:
import getpass
import os
from dotenv import load_dotenv

load_dotenv(override=True)

if (not os.environ.get("OPENAI_API_KEY")) or (os.environ.get("OPENAI_API_KEY")=='your-key-goes-here'):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

The next cell sets up a local language model as the default to create the embeddings if you don't have an OpenAI key. LlamaIndex' default is to use OpenAI, so not running the following cell will assume you want to continue with OpenAI.

In [6]:
from llama_index.legacy import ServiceContext, set_global_service_context
from llama_index.legacy.embeddings import LangchainEmbedding
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import FakeEmbeddings

# Patch llama_index.legacy to accept current OpenAI models
import llama_index.legacy.llms.openai_utils as _oai
_oai.ALL_AVAILABLE_MODELS.update({"gpt-4o": 128000, "gpt-4o-mini": 128000})
_oai.CHAT_MODELS.update({"gpt-4o": 128000, "gpt-4o-mini": 128000})

from llama_index.legacy.llms import OpenAI

if (not os.environ.get("OPENAI_API_KEY")) or (os.environ.get("OPENAI_API_KEY")=='your-key-goes-here'):
    lc_embed_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    service_context = ServiceContext.from_defaults(
        embed_model=LangchainEmbedding(lc_embed_model)
    )
else:
    service_context = ServiceContext.from_defaults(
        llm=OpenAI(model="gpt-4o-mini")
    )

set_global_service_context(service_context)

Now we'll configure the VectorStore object that will be used to save our vectors in IRIS.

In [7]:
from llama_index.legacy import StorageContext
from llama_index.legacy.indices.vector_store import VectorStoreIndex
from llama_iris import IRISVectorStore

# StorageContext captures how vectors will be stored
vector_store = IRISVectorStore.from_params(
    connection_string = url,
    table_name = "paul_graham_essay",
    embed_dim = 1536,  # openai embedding dimensionality
    # embed_dim = 384, # HugginFace all-MiniLM-L6-v2 dimensionality
    engine_args = { "connect_args": {"sslcontext": sslcontext} }
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

And now putting it all together: feeding the documents to our VectorStore.

In [8]:
index = VectorStoreIndex.from_documents(
    documents, 
    storage_context=storage_context, 
    show_progress=True, 
)
query_engine = index.as_query_engine()

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/22 [00:00<?, ?it/s]

In [9]:
# # If reconnecting to the vector store, use this: 

index = VectorStoreIndex.from_vector_store(vector_store=vector_store)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
query_engine = index.as_query_engine()

# Adding documents to existing index

for d in documents:
    index.insert(document=d, storage_context=storage_context)

In [10]:
response = query_engine.query("What did the author do?")

In [11]:
import textwrap
print(textwrap.fill(str(response), 100))

The author worked on writing and programming before college, initially focusing on short stories and
later transitioning to programming with early computers. They learned programming on an IBM 1401 and
later a TRS-80 microcomputer, creating simple games and a word processor. After starting college
with an interest in philosophy, they switched to studying artificial intelligence (AI) after finding
philosophy courses unengaging. The author taught themselves Lisp, reverse-engineered a program
called SHRDLU for their undergraduate thesis, and later pursued graduate studies in AI at Harvard.
They eventually recognized the limitations of AI as practiced at the time and shifted their focus to
Lisp, deciding to write a book about it while in graduate school.


In [12]:
response = query_engine.query("What happened in the mid 1980s?")
print(textwrap.fill(str(response), 100))

In the mid 1980s, there was a growing interest in artificial intelligence (AI), influenced by
cultural works such as a novel by Heinlein called "The Moon is a Harsh Mistress," which featured an
intelligent computer, and a PBS documentary showcasing Terry Winograd using a program called SHRDLU.
This period marked a shift in focus for some individuals, leading them to pursue AI despite the lack
of formal classes in the field at certain universities.
